## Configuration 

In [ ]:
# Configuration de récupération 
URI_API_BASE_MONGO_DB_LOCAL = "http://localhost:8080"
URI_API_BASE_MONGO_DB_AZURE = "https://api-base-fastapi-mpt-dkhjb0cydga2aydc.canadacentral-01.azurewebsites.net"
URI_API_REQUEST_GITHUB = "http://localhost:7000"


## Vérification du nombre d'éléments en base

In [ ]:
import requests

def appel_api_base_count_collection( url = URI_API_BASE_MONGO_DB_LOCAL):
    url = url+"/count"
    response = requests.get(url)
    print(f"code http: "+str(response.status_code))
    print(f"count : "+str(response.text))

appel_api_base_count_collection()

## Appel à l'API Rest github

In [ ]:
import requests

def get_call(paths,url, params=None, header=None):
    url = f"{url}/{paths}"
    response = requests.get(url, params=params, headers=header)
    if response.status_code == 200:
        return response.json()
    else:
        print(f"Error: {response.status_code} - {response.json().get('message', 'Unknown error')}")
        return None
    
repositories=get_call(paths="start",url="https://api-github-recup-data-auto-mpt-g4fpatffcxd7dkhd.canadacentral-01.azurewebsites.net")
print(repositories)

## Vérification de l'insertion des données

In [ ]:
import requests

def appel_api_base_count_collection( url = URI_API_BASE_MONGO_DB_LOCAL):
    url = url+"/count"
    response = requests.get(url)
    print(f"code http: "+str(response.status_code))
    print(f"count : "+str(response.text))

appel_api_base_count_collection()

## Visualisation d'une donnée type

In [ ]:
import requests

def appel_api_base_show_data(url = URI_API_BASE_MONGO_DB_LOCAL,page = 1,page_size = 10000):
    url = url+"/show_data?page="+str(page)+"&page_size="+str(page_size)
    response = requests.get(url)
    print(f"code http: "+str(response.status_code))
    return (response.text)

# Appel de l'API avec les bonnes dates
reponse = appel_api_base_show_data( url=URI_API_BASE_MONGO_DB_LOCAL)

## Entrainement du modèle à partir des données récupérées précédemment 

### Pré-traitement des données récupérées

In [ ]:
import pandas as pd
df_new = pd.DataFrame(repositories['data'])

df_new = df_new[['name','full_name','description','stargazers_count','watchers_count','forks_count','created_at','updated_at','language']]
df_new = df_new.dropna(subset=['created_at', 'language'])
df_new['created_at'] = df_new['created_at'].astype(str)  # S'assurer que c'est bien une string
df_new['created_at'] = df_new['created_at'].apply(lambda x: x + "Z" if not x.endswith("Z") else x)
df_new['created_at'] = pd.to_datetime(df_new['created_at'])
max_date = df_new['created_at'].max()  # La date la plus récente
recent_month = max_date.month  # Extraire le mois
recent_year = max_date.year  # Extraire l'année

# Filtrer le DataFrame pour ne garder que les données du mois le plus récent
df_recent = df_new[df_new['created_at'].dt.month == recent_month]
df_recent = df_recent[df_recent['created_at'].dt.year == recent_year]

# Afficher les résultats pour vérifier
df_recent


In [ ]:
try:
    df_old = pd.read_pickle("historique.pkl")
except FileNotFoundError:
    df_old = pd.DataFrame() 
df = pd.concat([df_old, df_new]).drop_duplicates(subset=['created_at'])
df.to_pickle("historique.pkl")
#Enlever tous les langages qui ne contiennent qu'un dépôt github
language_counts = df['language'].value_counts()
df['language'].value_counts()
df = df[df['language'].isin(language_counts[language_counts > 1].index)]
df['created_at'] = df['created_at'].astype(str)

### Affichage de la distibution des données par langage

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 5))
# sns.countplot(y=df["language"], order=df["language"].value_counts().index)
plt.pie(df['language'].value_counts(), labels=df['language'].value_counts().index, startangle= 90, autopct= '%.f%%')
plt.xlabel('occurence')
plt.ylabel('langage')
plt.title("Distribution des langages dans le dataset")
plt.show()

### Entrainement du modele

In [2]:
# Différents modèles proposés
MODELE_NAME = "gradient_boosting"
# MODELE_NAME = "random_forest"


In [ ]:

import requests

# URL de l'API
url = "http://localhost:8085/train"

# Exemple de données d'entrée
data = {
    "model_name": MODELE_NAME,
    "epochs": 10,
    "batch_size": 32,
    "metrics": {"lr": 0.01, "n_estimators": 100},
    "data": df.to_dict(orient='records')
}

# Envoyer la requête POST
response = requests.post(url, json=data)

# Afficher la réponse
print(response.status_code)
model_name, model_version = response.json()
print(f"Modèle {model_name} Version : {model_version} enregistré Modèle entraîné avec une accuracy de {acc}")

## Prédiction suite à l'entrainement

In [ ]:
import requests

URI_CHANGE_STAGING_MODEL= "localhost:8085"

def appel_api_model_change_staging( url = URI_CHANGE_STAGING_MODEL, path="/model-stage", model_name = model_name,version = model_version,stage = "Production"):
    url = f"http://{url}{path}?model_name={model_name}&version={version}&stage={stage}"
    response = requests.post(url)
    print(f"code http: "+str(response.status_code))
    print(f"count : "+str(response.text))

appel_api_model_change_staging()


In [ ]:

import requests

# URL de l'API
url = "http://localhost:8084/predict"

# Exemple de données d'entrée
data = {
    "model_name": model_name,
    "model_version": model_version,
    "data": df_recent.to_dict(orient = 'records')
}

# Envoyer la requête POST
response = requests.post(url, json=data)

# Afficher la réponse
print(response.status_code)
print(response.json())